## 1. Instalación de Dependencias y Librerías del Proyecto
En esta primera celda instalamos todos los paquetes externos necesarios para el proyecto. Esto incluye librerías de Procesamiento de Lenguaje Natural (NLP), herramientas cartográficas y conectores de bases de datos. Utilizaremos el parámetro `-q` (quiet) para realizar una instalación limpia sin saturar la pantalla de logs.

In [9]:
# Instalación de librerías para NLP, GIS, bases de datos e interfaz gráfica
!pip install pysentimiento folium pandas pyproj requests branca pymysql sqlalchemy gradio -q

# Descarga del modelo lingüístico optimizado en español para SpaCy
!python -m spacy download es_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 46.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Importación de Módulos e Inicialización de Modelos BERT
Una vez instalados los paquetes, importamos los módulos en el entorno de ejecución e instanciamos los analizadores basados en la arquitectura de Transformers (BERT). Cargaremos dos modelos de `pysentimiento` (uno para la polaridad del sentimiento y otro para la detección de emociones) y el pipeline gramatical de `SpaCy`.

In [10]:
import os
import re
import spacy
import pandas as pd
import numpy as np
import folium
import requests
from folium.plugins import MarkerCluster, HeatMap
from pyproj import Transformer
from pysentimiento import create_analyzer
from branca.element import Template, MacroElement

print("Cargando y configurando modelos de lenguaje (Transformers BERT)...")

# Inicialización de analizadores de texto en español
analyzer = create_analyzer(task="sentiment", lang="es")
emotion_analyzer = create_analyzer(task="emotion", lang="es")

# Carga del modelo de etiquetado gramatical (POS Tagging) de SpaCy
nlp = spacy.load("es_core_news_sm")
print("✓ Modelos listos para el análisis.")

Cargando y configurando modelos de lenguaje (Transformers BERT)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Modelos listos para el análisis.


## 3. Definición de Funciones de Limpieza y Lógica de Negocio (NLP)
Antes de procesar los conjuntos de datos, declaramos las funciones auxiliares que gobernarán la limpieza del texto, la normalización de nombres para el cruce de tablas, el filtrado de confianza de la IA (mínimo 60% de probabilidad) y la extracción de palabras clave morfológicas (Sustantivos y Adjetivos).

In [11]:
def limpiar_texto_encoding(texto):
    """Corrige problemas de decodificación entre Latin-1 y UTF-8."""
    if pd.isna(texto):
        return texto
    try:
        return str(texto).encode('latin-1').decode('utf-8')
    except:
        return str(texto)

def norm_name(texto):
    """Normaliza cadenas eliminando acentos, caracteres especiales y stop-words de alojamiento."""
    if pd.isna(texto):
        return ""
    texto = str(texto).lower().strip()
    texto = re.sub(r'[áäâà]', 'a', texto)
    texto = re.sub(r'[éëêè]', 'e', texto)
    texto = re.sub(r'[íïîì]', 'i', texto)
    texto = re.sub(r'[óöôò]', 'o', texto)
    texto = re.sub(r'[úüûù]', 'u', texto)
    texto = re.sub(r'\b(hotel|hostal|pension|camping|casa)\b', '', texto)
    return " ".join(re.sub(r'[^\w\s]', '', texto).split())

def analyze_sentiment(text):
    """Clasifica sentimiento con BERT. Requiere un umbral mínimo de confianza del 60%."""
    if pd.isna(text) or not str(text).strip():
        return ('NEU', 1.0)
    res = analyzer.predict(text)
    return (res.output, res.probas[res.output]) if res.probas[res.output] >= 0.60 else ('INCONCLUSO', 0.5)

def analyze_emotion(text):
    """Identifica la emoción predominante del texto y la traduce al castellano."""
    if pd.isna(text) or not str(text).strip():
        return 'NEUTRAL'
    try:
        res = emotion_analyzer.predict(text)
        traducciones = {
            'others': 'NEUTRAL', 'joy': 'ALEGRÍA', 'sadness': 'TRISTEZA',
            'anger': 'ENFADO', 'surprise': 'SORPRESA', 'disgust': 'ASCO', 'fear': 'MIEDO'
        }
        return traducciones.get(res.output.lower(), 'NEUTRAL')
    except:
        return 'NEUTRAL'

def extract_keywords(text):
    """Extrae las dos palabras clave más importantes (Sustantivos/Adjetivos) usando SpaCy."""
    doc = nlp(str(text))
    words = [t.text.lower() for t in doc if t.pos_ in ["NOUN", "ADJ"] and not t.is_stop]
    return ", ".join(list(set(words))[:2]).capitalize() if words else "General"

# 4. Ingesta, Reproyección Cartográfica y Fusión de Datasets (ETL)
En este bloque cargamos el archivo oficial OpenRTA y convertimos sus coordenadas planas UTM (proyección local EPSG:25830) a coordenadas esféricas GPS WGS84 (EPSG:4326). Posteriormente, cargamos el archivo de opiniones subjetivas y ejecutamos una fusión interna (`pd.merge`) utilizando las claves normalizadas creadas en el paso anterior.

In [12]:
try:
    # 1. Carga y filtrado por Almería del dataset catastral
    df_raw = pd.read_csv('dataset-openrta.csv', sep='|', encoding='latin-1', on_bad_lines='skip', low_memory=False)
    df_alm = df_raw[df_raw['provinces'].astype(str).str.contains('ALMER', case=False, na=False)].copy()

    # 2. Reparación de codificación de texto
    for col in ['name', 'municipalities', 'group', 'establishment_address']:
        if col in df_alm.columns:
            df_alm[col] = df_alm[col].apply(limpiar_texto_encoding)

    # 3. Transformación matemática de coordenadas (ED50/ETRS89 UTM 30N -> WGS84)
    transformer = Transformer.from_crs("epsg:25830", "epsg:4326")
    def convertir_coords(x, y):
        try:
            x_f, y_f = float(str(x).replace(',', '.')), float(str(y).replace(',', '.'))
            return transformer.transform(x_f, y_f)
        except:
            return None, None

    df_alm['lat'], df_alm['lon'] = zip(*df_alm.apply(lambda r: convertir_coords(r['coord_x'], r['coord_y']), axis=1))
    df_final = df_alm.dropna(subset=['lat', 'lon']).reset_index(drop=True)
    print(f"✓ {len(df_final)} establecimientos base geolocalizados correctamente.")

    # 4. Carga de opiniones y cruce matricial
    df_reviews = pd.read_csv('reseñas_subjetivas_almeria.csv', sep='|', encoding='utf-8')
    df_final['match_key'] = df_final['name'].apply(norm_name)
    df_reviews['match_key'] = df_reviews['establishment_name'].apply(norm_name)

    df_report = pd.merge(df_final, df_reviews, on='match_key', how='inner')
    print(f"Pipeline de fusión completado: {len(df_report)} registros vinculados.")

except Exception as e:
    print(f"Error crítico en el proceso de carga y transformación: {e}")
    df_report = pd.DataFrame()

✓ 441 establecimientos base geolocalizados correctamente.
Pipeline de fusión completado: 295 registros vinculados.


# 5. Ejecución del Enriquecimiento de Datos con Inteligencia Artificial
Con la base de datos unificada, procedemos a ejecutar de forma masiva los modelos BERT y los filtros morfológicos sobre los textos desestructurados de las opiniones. Adicionalmente, realizamos la discretización de variables continuas como la edad, segmentando a los usuarios en rangos de edad sociológicos.

In [13]:
if not df_report.empty:
    print("Inyectando analítica de Inteligencia Artificial en el DataFrame...")

    # Aplicación en lote de los modelos predictivos y de extracción
    df_report['label'], _ = zip(*df_report['review_text'].apply(analyze_sentiment))
    df_report['palabras_clave'] = df_report['review_text'].apply(extract_keywords)
    df_report['emocion_ia'] = df_report['review_text'].apply(analyze_emotion)

    # Clasificación demográfica en intervalos cerrados (Pandas Cut)
    df_report['rango_edad'] = pd.cut(
        df_report['edad'],
        bins=[0, 30, 55, 100],
        labels=['Joven (<30)', 'Adulto (30-55)', 'Mayor (>55)']
    )
    print(f"Dataset enriquecido con éxito. Estructura lista para visualización.")
else:
    print("DataFrame vacío. Revisa los pasos anteriores.")

Inyectando analítica de Inteligencia Artificial en el DataFrame...
Dataset enriquecido con éxito. Estructura lista para visualización.


# 6. Renderizado del Mapa Interactivo de Reputación Geográfica (Folium)
En esta última fase generamos la representación cartográfica interactiva. El mapa incluye una máscara perimetral oscura para centrar el foco en la provincia de Almería, agrupación dinámica de capas según el veredicto de sentimiento de la IA, personalización de iconos basada en la tipología del negocio y ventanas flotantes enriquecidas con HTML.

In [22]:
from IPython.display import display, HTML

if not df_report.empty:
    print("Construyendo mapa interactivo SentiAlmería...")
    df_mapa_perfecto = df_report.copy()
    col_nombre = [c for c in df_report.columns if any(p in c.lower() for p in ['name', 'nombre', 'establec', 'establish'])][0]

    # ==============================================================================
    # 1. INICIALIZACIÓN DEL MAPA BASE (CARTODB POSITRON)
    # ==============================================================================
    m = folium.Map(location=[37.1, -2.4], zoom_start=9, min_zoom=2, max_zoom=18, tiles="cartodbpositron")
    m.get_root().header.add_child(folium.Element('<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">'))

    # ==============================================================================
    # 2. CAPA GEOJSON: MÁSCARA PERIMETRAL ENFOQUE OSCURO
    # ==============================================================================
    try:
        geojson_url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/spain-provinces.geojson"
        geojson_data = requests.get(geojson_url).json()
        almeria_feat = [f for f in geojson_data['features'] if f['properties']['name'] == 'Almería'][0]
        mundo = [[90, -180], [90, 180], [-90, 180], [-90, -180], [90, -180]]
        coords_alm = almeria_feat['geometry']['coordinates'] if almeria_feat['geometry']['type'] == 'Polygon' else [p[0] for p in almeria_feat['geometry']['coordinates']]
        mascara_inv = {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [mundo] + (coords_alm if almeria_feat['geometry']['type'] == 'Polygon' else coords_alm)}}
        folium.GeoJson(mascara_inv, style_function=lambda x: {'fillColor': '#0f172a', 'color': '#0f172a', 'fillOpacity': 0.72}).add_to(m)
    except:
        pass

    # ==============================================================================
    # 3. CAPA DE DENSIDAD SIMULTÁNEA: MAPA DE CALOR (HEATMAP)
    # ==============================================================================
    heat_data = [[row['lat'], row['lon']] for idx, row in df_mapa_perfecto.iterrows() if not pd.isna(row['lat'])]
    HeatMap(heat_data, name="🔥 Mapa de Calor", show=False, radius=13, blur=10).add_to(m)

    # ==============================================================================
    # 4. CAPAS DE CONTROL POR SUBGRUPOS DE SENTIMIENTO INTERACTIVO
    # ==============================================================================
    hex_colors = {'POS': '#22c55e', 'NEG': '#991b1b', 'NEU': '#1e3a8a', 'INC': '#374151'}
    folium_colors = {'POS': 'green', 'NEG': 'red', 'NEU': 'blue', 'INC': 'gray'}
    subgrupos_senti = {k: folium.FeatureGroup(name=f"Sentimiento: {k}", show=(k!='INC')) for k in hex_colors.keys()}
    for sg in subgrupos_senti.values():
        sg.add_to(m)

    # ==============================================================================
    # 5. GENERACIÓN DINÁMICA DE MARCADORES CON CONDICIONAL DE TIPOLOGÍA
    # ==============================================================================
    for idx, row in df_mapa_perfecto.iterrows():
        lat, lon = row.get('lat'), row.get('lon')
        if pd.isna(lat) or pd.isna(lon):
            continue

        sentimiento = str(row.get('label', 'POS')).strip().upper()
        group_label = str(row.get('group', 'Alojamiento')).strip().lower()
        if sentimiento not in hex_colors:
            sentimiento = 'INC'

        if any(x in group_label for x in ['hotel', 'resort', 'apartahotel', 'vivienda', 'vut']):
            icono, label_t, img = 'bed', "Hotel", "https://images.unsplash.com/photo-1566073771259-6a8506099945?w=400"
        elif any(x in group_label for x in ['camping', 'campamento', 'caravana']):
            icono, label_t, img = 'tree', "Camping", "https://images.unsplash.com/photo-1504280390367-361c6d9f38f4?w=400"
        elif any(x in group_label for x in ['hostal', 'pension', 'albergue']):
            icono, label_t, img = 'home', "Hostal / Pensión", "https://images.unsplash.com/photo-1555854877-bab0e564b8d5?w=400"
        else:
            icono, label_t, img = 'key', "Alojamiento", "https://images.unsplash.com/photo-1469854523086-cc02fe5d8800?w=400"

        nombre_establecimiento = str(row.get(col_nombre, 'Establecimiento'))

        html_popup = f"""
        <div style="font-family: 'Segoe UI', sans-serif; width: 220px;">
            <img src="{img}" style="width:100%; height:90px; object-fit:cover; border-radius:6px; margin-bottom:6px;">
            <h5 style="margin:4px 0 2px 0; color:#0f172a; font-size:13px; font-weight:700;">{nombre_establecimiento}</h5>
            <small style="color:#64748b; font-weight:600; text-transform: uppercase;">{label_t} | {str(row.get('estrellas', '-'))} ★</small><br>
            <div style="background:{hex_colors[sentimiento]}; color:white; padding:3px 6px; border-radius:4px; display:inline-block; margin:6px 0; font-size:10px; font-weight:bold;">{sentimiento}</div>
            <p style="font-size:11px; margin:2px 0 0 0; font-style:italic; color:#334155; background:#f8fafc; padding:6px; border-radius:4px;">
                "{str(row.get('palabras_clave', 'General'))}"
            </p>
        </div>"""

        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(html_popup, max_width=260),
            tooltip=f"📊 {nombre_establecimiento}",
            icon=folium.Icon(color=folium_colors[sentimiento], icon=icono, prefix='fa')
        ).add_to(subgrupos_senti[sentimiento])

    # ==============================================================================
    # 6. INYECCIÓN DE LEYENDA CORPORATIVA Y CONTROL DE CAPAS
    # ==============================================================================
    folium.LayerControl(position='topright', collapsed=False).add_to(m)

    leyenda_html = f'''{{% macro html(this, kwargs) %}}
    <div style="position: fixed; bottom: 30px; left: 30px; width: 245px; background: white; border: 2px solid #e2e8f0; z-index: 9999; padding: 12px; border-radius: 12px; font-family: \\'Segoe UI\\'; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
        <b style="font-size: 13px; color: #0f172a;"><center>📍 SIMBOLOGÍA MAPA</center></b><hr style="margin: 6px 0;">
        <div style="font-size: 11px; display: flex; flex-direction: column; gap: 4px; font-weight: 600;">
            <div><i class="fa fa-circle" style="color: {hex_colors['POS']};"></i> Verde: POSITIVO</div>
            <div><i class="fa fa-circle" style="color: {hex_colors['NEG']};"></i> Rojo: NEGATIVO</div>
            <div><i class="fa fa-circle" style="color: {hex_colors['NEU']};"></i> Azul: NEUTRO</div>
        </div>
    </div>
    {{% endmacro %}}'''

    macro = MacroElement()
    macro._template = Template(leyenda_html)
    m.get_root().add_child(macro)

    # ==============================================================================
    # 7. FORZADO DE DESPLIEGUE GRÁFICO (SOLUCIÓN PARA CUADERNOS COLAB/JUPYTER)
    # ==============================================================================
    print("Renderizando interfaz cartográfica interactiva...")
    display(m)

else:
    print("DataFrame vacío. Comprueba el flujo.")

Construyendo mapa interactivo SentiAlmería...
Renderizando interfaz cartográfica interactiva...


# 7. Persistencia de Datos: Exportación a Amazon Web Services (AWS RDS)
En este bloque cerramos la fase de almacenamiento de nuestro pipeline de datos (ETL). Una vez que los datos han sido limpiados, georreferenciados y enriquecidos con los modelos predictivos BERT (Inteligencia Artificial), utilizamos la librería `SQLAlchemy` junto con el conector `PyMySQL` para inyectar de forma masiva el DataFrame resultante dentro de nuestra base de datos relacional MySQL alojada en una instancia de Amazon RDS (AWS). Esto permitirá que herramientas de Inteligencia de Negocio como Power BI consuman los datos directamente y en tiempo real.

In [19]:
import pandas as pd
import numpy as np
import re
from sqlalchemy import create_engine, text

print("INICIANDO FILTRADO")

# ==============================================================================
# 1. PARÁMETROS DE INFRAESTRUCTURA DE RED (AWS CLOUD CONFIGURATION)
# ==============================================================================
USUARIO = "admin_almeria"
CONTRASENA = "Almeria2026*"
HOST = "almeria-viajes-db.ci8r4guxz0rk.us-east-1.rds.amazonaws.com"
PUERTO = "3306"
DB_NAME = "almeria_viajes_db"

def norm_name(t):
    """
    Normalización estandarizada de cadenas de texto (Fuzzy Matching Key).
    Elimina acentos, diéresis, caracteres especiales y stop-words categóricas.
    """
    if pd.isna(t): return ""
    t = str(t).lower().strip()
    t = re.sub(r'[áäâà]', 'a', t); t = re.sub(r'[éëêè]', 'e', t); t = re.sub(r'[íïîì]', 'i', t)
    t = re.sub(r'[óöôò]', 'o', t); t = re.sub(r'[úüûù]', 'u', t)
    t = re.sub(r'\b(hotel|hostal|pension|camping|casa)\b', '', t)
    return " ".join(re.sub(r'[^\w\s]', '', t).split())

try:
    # ==============================================================================
    # 2. INGESTA DE LAS FUENTES DE DATOS PRIMARIAS
    # ==============================================================================
    df_rta = pd.read_csv('dataset-openrta.csv', sep='|', encoding='utf-8', engine='python', on_bad_lines='skip')
    df_rev = pd.read_csv('reseñas_subjetivas_almeria.csv', sep='|', encoding='utf-8')

    # ==============================================================================
    # 3. CRUCE MATRICIAL Y CORRELACIÓN DE REGISTROS
    # ==============================================================================
    df_rta['match_key'] = df_rta['name'].apply(norm_name)
    df_rev['match_key'] = df_rev['establishment_name'].apply(norm_name)
    df_unificado = pd.merge(df_rta, df_rev, on='match_key', how='inner')

    # ==============================================================================
    # 4. SEGMENTACIÓN DEMOGRÁFICA DISCRETA
    # ==============================================================================
    df_unificado['rango_edad'] = pd.cut(
        df_unificado['edad'],
        bins=[0, 30, 55, 100],
        labels=['Joven (<30)', 'Adulto (30-55)', 'Mayor (>55)']
    )

    # ==============================================================================
    # 5. HOMOLOGACIÓN DE ESQUEMAS Y REGLAS DE LIMPIEZA (DATA CLEANING)
    # ==============================================================================
    # Diccionario de mapeo para adaptar las variables al modelo de datos corporativo
    mapeo_columnas = {
        'name': 'nombre_establecimiento',
        'locality': 'municipio',
        'estrellas': 'categoria_alojamiento',
        'plataforma': 'plataforma_origen',
        'review_text': 'comentario_turista',
        'edad': 'edad',
        'sexo': 'sexo',
        'rango_edad': 'rango_edad',
        'categoria_sentimiento': 'sentimiento_ia'
    }

    # Filtrado selectivo de columnas requeridas y eliminación de redundancias
    df_final_aws = df_unificado[list(mapeo_columnas.keys())].copy()
    df_final_aws.rename(columns=mapeo_columnas, inplace=True)
    df_final_aws = df_final_aws.loc[:, ~df_final_aws.columns.duplicated()]

    # Aplicación de filtros de integridad: Eliminación estricta de registros nulos o vacíos
    # en las dimensiones clave del negocio ('nombre_establecimiento' y 'municipio')
    df_final_aws.dropna(subset=['nombre_establecimiento', 'municipio'], inplace=True)
    df_final_aws = df_final_aws[df_final_aws['municipio'].str.strip() != ""]
    df_final_aws = df_final_aws[df_final_aws['nombre_establecimiento'].str.strip() != ""]

    print(f"Filtro de solidez completado. Nos quedamos solo con las filas que tienen todos sus datos perfectos.")

    # ==============================================================================
    # 6. EXPORTACIÓN MASIVA E INYECCIÓN EN AMAZON RDS
    # ==============================================================================
    cadena_conexion = f"mysql+pymysql://{USUARIO}:{CONTRASENA}@{HOST}:{PUERTO}/{DB_NAME}"
    engine_proyecto = create_engine(cadena_conexion)

    print(f"Inyectando filas filtradas en la nube de AWS...")
    # 'if_exists="replace"': Sobrescribe la tabla garantizando un esquema actualizado sin colisiones de duplicados
    df_final_aws.to_sql(name='informe_turismo', con=engine_proyecto, if_exists='replace', index=False)

    # ==============================================================================
    # 7. AUDITORÍA Y VERIFICACIÓN FÍSICA EN EL SERVIDOR CLOUD
    # ==============================================================================
    # Establecemos una conexión directa para auditar el volumen real de datos inyectados en AWS
    with engine_proyecto.connect() as conexion:
        resultado = conexion.execute(text("SELECT COUNT(*) FROM informe_turismo"))
        filas_en_aws = resultado.fetchone()[0]

    print("\n" + "="*50)
    print(f"¡DESPLIEGUE COMPLETADO!")
    print(f"Columnas listas: {list(df_final_aws.columns)}")
    print(f"Filas sólidas confirmadas: {filas_en_aws} filas.")
    print("="*50)

except Exception as e:
    print(f"FALLO CRÍTICO: {e}")

INICIANDO FILTRADO
Filtro de solidez completado. Nos quedamos solo con las filas que tienen todos sus datos perfectos.
Inyectando filas filtradas en la nube de AWS...

¡DESPLIEGUE COMPLETADO!
Columnas listas: ['nombre_establecimiento', 'municipio', 'categoria_alojamiento', 'plataforma_origen', 'comentario_turista', 'edad', 'sexo', 'rango_edad', 'sentimiento_ia']
Filas sólidas confirmadas: 103 filas.


# 8. Generación y Exportación del Sistema Cartográfico Autónomo (Fase de Distribución)
En esta fase del pipeline, el motor cartográfico procesa de forma interna las dimensiones geoespaciales unificadas, proyectando las capas de calor analíticas y la iconografía adaptativa por tipología de alojamiento turístico.

Con el objetivo de garantizar una arquitectura de desarrollo limpia y optimizar los recursos de memoria del entorno de ejecución, **el mapa no se renderiza dentro del cuaderno de Google Colab**. En su lugar, el sistema compila toda la infraestructura visual y la exporta directamente a un archivo estático independiente denominado `SentiAlmeria_Mapa_Clientes.html`.

Este entregable autónomo puede distribuirse a los clientes finales para que lo abran con un simple doble clic en cualquier navegador web, permitiendo una visualización fluida a pantalla completa y manteniendo el entorno de Colab exclusivamente como la consola analítica del proyecto.

In [28]:
if not df_report.empty:
    print("Iniciando la compilación del mapa interactivo SentiAlmería...")
    df_mapa_perfecto = df_report.copy()
    col_nombre = [c for c in df_report.columns if any(p in c.lower() for p in ['name', 'nombre', 'establec', 'establish'])][0]

    # ==============================================================================
    # 1. INSTANCIACIÓN DEL ESPACIO CARTOGRÁFICO BASE
    # ==============================================================================
    m = folium.Map(location=[37.1, -2.4], zoom_start=9, min_zoom=2, max_zoom=18, tiles="cartodbpositron")
    m.get_root().header.add_child(folium.Element('<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">'))

    # ==============================================================================
    # 2. CAPA GEOJSON: MÁSCARA MUNICIPAL / PROVINCIAL DE ENFOQUE
    # ==============================================================================
    try:
        geojson_url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/spain-provinces.geojson"
        geojson_data = requests.get(geojson_url).json()
        almeria_feat = [f for f in geojson_data['features'] if f['properties']['name'] == 'Almería'][0]
        mundo = [[90, -180], [90, 180], [-90, 180], [-90, -180], [90, -180]]
        coords_alm = almeria_feat['geometry']['coordinates'] if almeria_feat['geometry']['type'] == 'Polygon' else [p[0] for p in almeria_feat['geometry']['coordinates']]
        mascara_inv = {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [mundo] + (coords_alm if almeria_feat['geometry']['type'] == 'Polygon' else coords_alm)}}
        folium.GeoJson(mascara_inv, style_function=lambda x: {'fillColor': '#0f172a', 'color': '#0f172a', 'fillOpacity': 0.72}).add_to(m)
    except:
        pass

    # ==============================================================================
    # 3. CAPA DE DENSIDAD ESPACIAL (HEATMAP LAYER)
    # ==============================================================================
    heat_data = [[row['lat'], row['lon']] for idx, row in df_mapa_perfecto.iterrows() if not pd.isna(row['lat'])]
    HeatMap(heat_data, name="🔥 Mapa de Calor", show=False, radius=13, blur=10).add_to(m)

    # ==============================================================================
    # 4. CAPAS DE CONTROL ANALÍTICO (FEATURE GROUPS)
    # ==============================================================================
    hex_colors = {'POS': '#22c55e', 'NEG': '#991b1b', 'NEU': '#1e3a8a', 'INC': '#374151'}
    folium_colors = {'POS': 'green', 'NEG': 'red', 'NEU': 'blue', 'INC': 'gray'}
    subgrupos_senti = {k: folium.FeatureGroup(name=f"Sentimiento: {k}", show=(k!='INC')) for k in hex_colors.keys()}
    for sg in subgrupos_senti.values():
        sg.add_to(m)

    # ==============================================================================
    # 5. INYECCIÓN DINÁMICA DE MARCADORES E ICONOGRAFÍA BUSINESS
    # ==============================================================================
    for idx, row in df_mapa_perfecto.iterrows():
        lat, lon = row.get('lat'), row.get('lon')
        if pd.isna(lat) or pd.isna(lon):
            continue

        sentimiento = str(row.get('label', 'POS')).strip().upper()
        group_label = str(row.get('group', 'Alojamiento')).strip().lower()
        if sentimiento not in hex_colors:
            sentimiento = 'INC'

        if any(x in group_label for x in ['hotel', 'resort', 'apartahotel', 'vivienda', 'vut']):
            icono, label_t, img = 'bed', "Hotel / Apto", "https://images.unsplash.com/photo-1566073771259-6a8506099945?w=400"
        elif any(x in group_label for x in ['camping', 'campamento', 'caravana']):
            icono, label_t, img = 'tree', "Camping", "https://images.unsplash.com/photo-1504280390367-361c6d9f38f4?w=400"
        elif any(x in group_label for x in ['hostal', 'pension', 'albergue']):
            icono, label_t, img = 'home', "Hostal / Pensión", "https://images.unsplash.com/photo-1555854877-bab0e564b8d5?w=400"
        else:
            icono, label_t, img = 'key', "Alojamiento", "https://images.unsplash.com/photo-1469854523086-cc02fe5d8800?w=400"

        nombre_establecimiento = str(row.get(col_nombre, 'Establecimiento'))

        html_popup = f"""
        <div style="font-family: 'Segoe UI', sans-serif; width: 220px;">
            <img src="{img}" style="width:100%; height:90px; object-fit:cover; border-radius:6px; margin-bottom:6px;">
            <h5 style="margin:4px 0 2px 0; color:#0f172a; font-size:13px; font-weight:700;">{nombre_establecimiento}</h5>
            <small style="color:#64748b; font-weight:600; text-transform: uppercase;">{label_t} | {str(row.get('estrellas', '-'))} ★</small><br>
            <div style="background:{hex_colors[sentimiento]}; color:white; padding:3px 6px; border-radius:4px; display:inline-block; margin:6px 0; font-size:10px; font-weight:bold;">{sentimiento}</div>
            <p style="font-size:11px; margin:2px 0 0 0; font-style:italic; color:#334155; background:#f8fafc; padding:6px; border-radius:4px;">
                "{str(row.get('palabras_clave', 'General'))}"
            </p>
        </div>"""

        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(html_popup, max_width=260),
            tooltip=f"📊 {nombre_establecimiento}",
            icon=folium.Icon(color=folium_colors[sentimiento], icon=icono, prefix='fa')
        ).add_to(subgrupos_senti[sentimiento])

    # ==============================================================================
    # 6. COMPILACIÓN DE COMPONENTES ADICIONALES (LEYENDAS Y CAPAS)
    # ==============================================================================
    folium.LayerControl(position='topright', collapsed=False).add_to(m)

    leyenda_html = f'''{{% macro html(this, kwargs) %}}
    <div style="position: fixed; bottom: 30px; left: 30px; width: 245px; background: white; border: 2px solid #e2e8f0; z-index: 9999; padding: 12px; border-radius: 12px; font-family: \\'Segoe UI\\'; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
        <b style="font-size: 13px; color: #0f172a;"><center>📍 SIMBOLOGÍA MAPA</center></b><hr style="margin: 6px 0;">
        <div style="font-size: 11px; display: flex; flex-direction: column; gap: 4px; font-weight: 600;">
            <div><i class="fa fa-circle" style="color: {hex_colors['POS']};"></i> Verde: POSITIVO</div>
            <div><i class="fa fa-circle" style="color: {hex_colors['NEG']};"></i> Rojo: NEGATIVO</div>
            <div><i class="fa fa-circle" style="color: {hex_colors['NEU']};"></i> Azul: NEUTRO</div>
        </div>
    </div>
    {{% endmacro %}}'''

    macro = MacroElement()
    macro._template = Template(leyenda_html)
    m.get_root().add_child(macro)

    # ==============================================================================
    # 7. EXPORTACIÓN A ARCHIVO INDEPENDIENTE (SISTEMA LIMPIO)
    # ==============================================================================
    nombre_archivo = "SentiAlmeria_Mapa_Clientes.html"
    m.save(nombre_archivo)

    print("\n" + "="*60)
    print(f"¡MAPA DE REPUTACIÓN EXPORTADO CON ÉXITO!")
    print(f"Archivo generado: '{nombre_archivo}'")
    print(f"Estado: Listo para entrega independiente (Pantalla Completa Nativa).")
    print("="*60)

else:
    print("DataFrame vacío. Comprueba el flujo.")

Iniciando la compilación del mapa interactivo SentiAlmería...

¡MAPA DE REPUTACIÓN EXPORTADO CON ÉXITO!
Archivo generado: 'SentiAlmeria_Mapa_Clientes.html'
Estado: Listo para entrega independiente (Pantalla Completa Nativa).


# 9. Despliegue del Detector de Sentimientos y Auditoría en Vivo (Gradio GUI)
Para la fase de demostración del funcionamiento de la Inteligencia Artificial dentro de la celda de Colab, levantamos la interfaz clásica de `Gradio`. Este componente actúa exclusivamente como una consola interactiva de pruebas. Permite introducir textos o reseñas libres para evaluar los tiempos de respuesta e inferencia de los modelos Transformer BERT y las reglas morfológicas de SpaCy, devolviendo las etiquetas analíticas correspondientes sin interferir con el visor geográfico.

In [26]:
import gradio as gr

def clasificador_interactivo(comentario):
    """Función de inferencia para conectar los campos de entrada con los predictores BERT."""
    if not comentario.strip():
        return "Por favor, introduce una reseña válida.", "N/A", "N/A"

    # Inferencia masiva utilizando los analizadores globales cargados previamente
    senti_pred, _ = analyze_sentiment(comentario)
    emocion_pred = analyze_emotion(comentario)
    keywords_pred = extract_keywords(comentario)

    # Diccionario estandarizado de visualización ejecutiva
    senti_map = {'POS': '🟢 POSITIVO', 'NEG': '🔴 NEGATIVO', 'NEU': '🔵 NEUTRO', 'INCONCLUSO': '⚫ INCONCLUSO'}
    return senti_map.get(senti_pred, senti_pred), emocion_pred, keywords_pred

print("Lanzando consola interactiva de Inteligencia Artificial...")

# Instanciación directa de la interfaz clásica de control
demo = gr.Interface(
    fn=clasificador_interactivo,
    inputs=gr.Textbox(lines=4, placeholder="Escribe la reseña de un establecimiento para auditar la IA...", label="Comentario del Usuario"),
    outputs=[
        gr.Textbox(label="Análisis de Sentimiento (IA BERT)"),
        gr.Textbox(label="Emoción Predominante Detectada"),
        gr.Textbox(label="Palabras Clave Extraídas (SpaCy POS)")
    ],
    title="🧠 SentiAlmería AI - Consola de Inferencia Semántica",
    description="Herramienta de auditoría para verificar la precisión del modelo en la clasificación de opiniones turísticas.",
    theme="soft"
)

demo.launch(share=True)

Lanzando consola interactiva de Inteligencia Artificial...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0778201ef4f0df268f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
